# BizLens v2.2.12 - Process Mining (Enriched)

Analyze **business processes** using native BizLens + **advanced pm4py** features (DFG, Process Map, Conformance).

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/solutiongate-learn/bizlens.git", "plotly"])
print("✅ BizLens v2.2.12 + pm4py + Plotly ready!")

In [ ]:
import bizlens as bl
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Optional: Enable timing
# bl.set_profiling(True)

## Dataset: Support Ticket Workflow

In [ ]:
np.random.seed(42)
events = []
base_time = datetime(2024, 1, 1)

for case_id in range(1, 101):
    activities = ['Create', 'Assign', 'Investigate', 'Resolve', 'Close']
    current_time = base_time + timedelta(days=case_id)
    for activity in activities:
        events.append({
            'case_id': f'TICKET-{case_id:03d}',
            'activity': activity,
            'timestamp': current_time,
            'agent': np.random.choice(['Alice', 'Bob', 'Charlie'])
        })
        current_time += timedelta(hours=np.random.gamma(2, 2))

event_log = pd.DataFrame(events)
print(f"Events: {len(event_log):,} | Cases: {event_log['case_id'].nunique():,}")

## BizLens Native Analysis

In [ ]:
bl.process_mining.case_metrics(event_log)
bl.process_mining.activity_metrics(event_log)
bl.process_mining.bottleneck_analysis(event_log)
bl.process_mining.variant_discovery(event_log)[0]
bl.process_mining.resource_analysis(event_log, resource_col='agent')

## Advanced: pm4py Integration

In [ ]:
import pm4py

# Convert to pm4py format
log = pm4py.formatters.format_dataframe(event_log, case_id='case_id', activity_key='activity', timestamp_key='timestamp')

# 1. Directly-Follows Graph (DFG)
dfg, start_activities, end_activities = pm4py.discovery.discover_dfg(log)
pm4py.vis.view_dfg(dfg, start_activities, end_activities)

# 2. Process Map (more readable)
process_map = pm4py.discovery.discover_process_map(log)
pm4py.vis.view_process_map(process_map)

print("✅ DFG and Process Map generated using pm4py")

## Conformance Checking (Optional)

In [ ]:
try:
    # Simple reference model (you can extend this)
    net, im, fm = pm4py.discovery.discover_petri_net_inductive(log)
    fitness = pm4py.conformance.fitness_token_based_replay(log, net, im, fm)
    print(f"Fitness (Token-based Replay): {fitness['perc_fit_traces']:.1f}%")
except Exception as e:
    print(f"Conformance check skipped: {e}")